# Assessment #1: Titanic Survival Classification

## 🎯 Task
This notebook trains an XGBoost classifier to predict Titanic passenger survival.

**Your goal**: Find and fix all bugs to make the notebook run successfully.

## ⏱️ Time: 10 minutes

## 📋 Expected Output
- Model trains successfully
- Predictions are generated
- Accuracy score is calculated

## 💡 Tips
- Run each cell sequentially
- Read error messages carefully
- Think about data science best practices
- There are 2 critical bugs that prevent execution and 2 subtle bugs


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import xgbost as xgb  # BUG #1 (CRITICAL): Typo - should be 'xgboost'


## 2. Load Titanic Dataset


In [ ]:
# Load dataset from HuggingFace
dataset = load_dataset("scikit-learn/titanic")
df = pd.DataFrame(dataset['train'])

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()


## 3. Data Exploration


In [ ]:
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nTarget Distribution:")
print(df['survived'].value_counts())


## 4. Feature Engineering


In [ ]:
# Select features for modeling
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
target = 'survived'

# Create working dataframe
df_model = df[features + [target]].copy()

# Encode categorical variable
df_model['sex'] = df_model['sex'].map({'male': 0, 'female': 1})

# Fill missing values with median
df_model['age'].fillna(df_model['age'].median(), inplace=True)
df_model['fare'].fillna(df_model['fare'].median(), inplace=True)

# BUG #2 (CRITICAL): Typo in column name when selecting features
# 'farex' doesn't exist - should be 'fare'
X_features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'farex']

print(f"Features for modeling: {features}")
print(f"\nData shape: {df_model.shape}")
df_model.head()


## 5. Feature Scaling


In [ ]:
# Separate features and target - using the typo variable!
X = df_model[X_features]  # This will fail - 'farex' doesn't exist
y = df_model[target]

# BUG #3 (SUBTLE): Data Leakage - Scaling before train/test split
# Scale features BEFORE split - This is wrong!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X_features)

print("Features scaled")
print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y.shape}")


## 6. Train/Test Split


In [ ]:
# BUG #4 (SUBTLE): Missing stratify parameter
# For imbalanced datasets, should use stratify=y to maintain class distribution

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


## 7. Train XGBoost Model


In [ ]:
# Initialize and train model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

# Train model
model.fit(X_train, y_train)

print("Model trained successfully!")


## 8. Make Predictions and Evaluate


In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test))


## 9. Feature Importance


In [ ]:
# Display feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)


## ✅ Success!

If you see this message, you've successfully debugged the notebook!

**Bugs you should have found:**
1. **CRITICAL**: Import typo (`xgbost` → `xgboost`)
2. **CRITICAL**: Column name typo (`farex` → `fare`)
3. **SUBTLE**: Data leakage (scaling before train/test split)
4. **SUBTLE**: Missing stratify parameter in train_test_split
